# 🍃 Exercise 1 · Your GPU workspace and a lab-trained baseline
**Operation CropGuard** · Kubeflow tutorial, part 1 of 5

## The mission
A farm app should diagnose a leaf from a phone photo. The best public training photos (**PlantVillage**) were taken in a lab, on plain backdrops. Models trained on them look great on lab photos.

**Will they still work in a real field?** You will find out.

## What you will do
1. Look at your workspace: the GPU, CPUs and data volume Kubeflow gave you.
2. Check the data on your volume (no downloads).
3. Fine-tune **ConvNeXt-Tiny** on lab photos. This is the **baseline**.
4. Test it on lab photos, then on **real phone photos of coffee leaves** (**RoCoLe**) it has never seen.
5. Try one manual fix (data augmentation) and measure whether it helps.

You finish with a baseline, an honest view of where it fails, and a feel for how slow and noisy manual tuning is. The next exercises automate that.

## The two datasets
| | PlantVillage (train) | RoCoLe (field test) |
|---|---|---|
| Photos | 7,629 leaves on plain lab backdrops | 1,560 phone photos from a real coffee plantation |
| Labels | 38 classes (crop + condition) | healthy, coffee rust, red spider mite |
| Used for | train / validate / test | test only, never trained on |
| Licence | CC BY-SA 3.0 (per its Hugging Face card) | CC BY 4.0 |

Coffee is not in PlantVillage, so the field test asks one simple question: **healthy or sick?** The 38-class model gives a **sick score**: `1 − (probability of the 12 healthy classes)`.

## How to work
Run the cells top to bottom with `Shift+Enter`. You need PyTorch and torchvision (CUDA build), numpy, pandas, matplotlib, scikit-learn and pillow. The whole notebook takes a few minutes of compute.

> 🎛️ **The knobs cell.** The next code cell, under "Setup", holds the settings you can change: epochs, learning rate, seed, augmentation recipe and more. When this notebook says *"re-run from the knobs cell"*, change a value there, then run that cell and every cell below it.

> ✋ **Predict**: guess before you run. 🔧 **Try it**: optional. 💡 **Why**: the reason. ✅ **Checkpoint**: what you should see.

## Setup: imports and the knobs

> 🎛️ **This is the knobs cell.** Change settings here, then run this cell and everything below it.

In [ ]:
import os, io, json, time, math, random, shutil, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
from PIL import Image, ImageFilter, ImageEnhance
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.max_colwidth', None)      # do not cut long row labels with '...'

# ---- Knobs. The defaults are what the instructors tested. -----------------------------------------------------------
SEED            = 42
EPOCHS          = 4                      # passes over the training photos
BATCH_SIZE      = 64                     # lower it (32, 16) if the GPU runs out of memory
LR              = 2e-4                   # peak learning rate (AdamW, warm-up then cosine decay)
WEIGHT_DECAY    = 0.05
LABEL_SMOOTHING = 0.1
RECIPE          = 'flip+rrc0.1+photo1'   # the augmentation recipe from step 8 (explained there)
NUM_WORKERS     = None                   # CPU processes that prepare batches (None = decide from the CPUs and shared memory this pod really has)
QUICK           = False                  # True = tiny smoke-test run (~1 min) to check that everything works

random.seed(SEED); np.random.seed(SEED); _ = torch.manual_seed(SEED)

---
## Step 1 · Your workspace
This notebook runs in a **pod**: a slice of a machine (CPUs, memory and **one GPU**) plus your **volumes**, all set up by Kubeflow. Let's see what you got. First the GPU, as the driver sees it:

In [ ]:
!nvidia-smi

Now the same GPU as **PyTorch** sees it. This is the check you will run at the start of every GPU session in your career.

In [ ]:
CUDA_OK = torch.cuda.is_available()
DEVICE = 'cuda' if CUDA_OK else 'cpu'
print(f'PyTorch {torch.__version__} | built for CUDA {torch.version.cuda} | cuDNN {torch.backends.cudnn.version()}')
if CUDA_OK:
    p = torch.cuda.get_device_properties(0)
    print(f'GPU 0: {p.name} | compute capability {p.major}.{p.minor} | {p.total_memory / 2**30:.1f} GiB | GPUs visible to this notebook: {torch.cuda.device_count()}')
    torch.backends.cudnn.benchmark = True        # let cuDNN pick the fastest convolution kernels for our fixed input size
else:
    print('⚠️  No GPU visible to PyTorch. This notebook switches to QUICK mode so it still runs, but the numbers will be meaningless.')
    print('    On the tutorial cluster this means the notebook server was created without a GPU: ask an instructor.')
    QUICK = True

**Why does the GPU matter?** A quick race: the same matrix multiplication (the core operation of neural networks) on the CPU and on the GPU.

In [ ]:
def matmul_tflops(device, dtype, n=2048, reps=20):
    a = torch.randn(n, n, device=device, dtype=dtype); b = torch.randn(n, n, device=device, dtype=dtype)
    for _ in range(3): a @ b                                   # warm-up
    if device == 'cuda': torch.cuda.synchronize()             # GPU calls are asynchronous: wait for them before reading the clock
    t0 = time.time()
    for _ in range(reps): a @ b
    if device == 'cuda': torch.cuda.synchronize()
    return 2 * n**3 * reps / (time.time() - t0) / 1e12

cpu = matmul_tflops('cpu', torch.float32)
print(f'CPU  fp32 : {cpu:7.2f} TFLOP/s')
if CUDA_OK:
    g32, g16 = matmul_tflops('cuda', torch.float32), matmul_tflops('cuda', torch.float16)
    print(f'GPU  fp32 : {g32:7.2f} TFLOP/s   ({g32 / cpu:,.0f}x the CPU)')
    print(f'GPU  fp16 : {g16:7.2f} TFLOP/s   ({g16 / cpu:,.0f}x the CPU)  <- tensor cores; this is why we train in mixed precision')

> 💡 **Why fp16?** GPUs run 16-bit maths much faster and use half the memory. PyTorch's `autocast` uses 16-bit where it is safe, and a `GradScaler` stops small gradients rounding to zero. The T4 supports fp16 but not bf16, so we use fp16.

Next: the CPUs, memory and volumes in your pod.

In [ ]:
def cgroup(path):
    try: return Path(path).read_text().strip()
    except Exception: return 'n/a'

def available_cpus():
    """CPU cores this pod may really use: the cgroup quota if one is set (Kubernetes CPU limit), otherwise the cores the OS shows."""
    try:
        quota, period = cgroup('/sys/fs/cgroup/cpu.max').split()
        if quota != 'max': return max(1, math.ceil(int(quota) / int(period)))
    except Exception: pass
    return len(os.sched_getaffinity(0))

CPUS = available_cpus()
SHM_MB = shutil.disk_usage('/dev/shm').total / 2**20 if Path('/dev/shm').exists() else None     # DataLoader workers hand batches to the trainer through shared memory
PREFETCH = 4                                                                                      # batches each worker keeps ready
if NUM_WORKERS is None: NUM_WORKERS = min(8, max(1, CPUS - 1))                                    # leave one core for the notebook itself
if SHM_MB is not None and SHM_MB < 512:
    NUM_WORKERS, PREFETCH = min(NUM_WORKERS, 2), 2
    print(f'⚠️  shared memory (/dev/shm) is only {SHM_MB:,.0f} MB: using {NUM_WORKERS} data-loader workers and a short prefetch so batches fit. (Kubeflow: tick "enable shared memory" when creating a notebook server.)')
print('CPU cores this notebook may use :', CPUS, '| cgroup cpu.max:', cgroup('/sys/fs/cgroup/cpu.max'))
print('memory limit (cgroup)           :', cgroup('/sys/fs/cgroup/memory.max'), '(bytes; "max" = no limit set)')
print('shared memory (/dev/shm)        :', f'{SHM_MB:,.0f} MB' if SHM_MB is not None else 'n/a')
print('data-loader workers we will use :', NUM_WORKERS)
!df -hT -x tmpfs -x devtmpfs -x squashfs -x overlay 2>/dev/null

> ✅ **Checkpoint 1.** `CUDA_OK` is `True`, the GPU name shows, and fp16 is several times faster than fp32. Your volumes appear in the `df` table.
>
> 🔧 **Try it.** Open a terminal (File → New → Terminal) and run `watch -n 1 nvidia-smi`. You will see the GPU wake up in Step 5.

---
## Step 2 · Check the data
The dataset sits on a **PersistentVolumeClaim (PVC)** mounted in your pod, so nobody downloads gigabytes over the venue Wi-Fi. The cell below looks for it. If it is mounted elsewhere, set `CROPGUARD_DATA` or add the path to `CANDIDATES`.

In [ ]:
CANDIDATES = [os.environ.get('CROPGUARD_DATA'), '/data/cropguard', '/home/jovyan/data/cropguard', '../data/cropguard', 'data/cropguard']
DATA = next((Path(c) for c in CANDIDATES if c and (Path(c) / 'MANIFEST.json').exists()), None)
if DATA is None:
    raise FileNotFoundError('Could not find the CropGuard data volume (a folder containing MANIFEST.json). Looked in: '
                            + ', '.join(str(c) for c in CANDIDATES if c) + '. Ask an instructor where the PVC is mounted, then set CROPGUARD_DATA.')
MANIFEST = json.loads((DATA / 'MANIFEST.json').read_text())
print('data volume:', DATA.resolve(), f"| {MANIFEST['total_gb']} GB | built {MANIFEST['built']}")
print(pd.Series({k: f'{v / 1e6:,.0f} MB' for k, v in MANIFEST['bytes'].items()}).to_string())

It holds two image arrays, two small tables and the model weights. Loading them also shows how fast your volume is (a second read is faster because the OS caches the file).

In [ ]:
t0 = time.time()
pv = pd.read_csv(DATA / 'plantvillage/meta.csv', index_col='idx'); ro = pd.read_csv(DATA / 'rocole/meta.csv', index_col='idx')
pv_img = np.load(DATA / 'plantvillage/images.npy'); ro_img = np.load(DATA / 'rocole/images.npy')
mb = (pv_img.nbytes + ro_img.nbytes) / 1e6; dt = time.time() - t0
print(f'read {mb:,.0f} MB from the volume in {dt:.1f}s ({mb / dt:,.0f} MB/s)')

CLASSES = MANIFEST['classes']; N_CLASSES = len(CLASSES); HEALTHY = np.array(MANIFEST['healthy_class_ids'])
y = pv['cls'].map({c: i for i, c in enumerate(CLASSES)}).values                           # class name -> integer label
TRAIN, VAL, TEST = [np.where(pv['split'] == s)[0] for s in ('train', 'val', 'test')]
ro_sick = ro['sick'].values.astype(int); plants = ro['plant'].values
DEV, LOCKED = [(ro['half'] == h).values for h in ('dev', 'locked')]
if QUICK: TRAIN = np.random.RandomState(SEED).permutation(TRAIN)[:512]; EPOCHS = 1      # smoke-test mode

print(f'\nPlantVillage: {len(pv):,} photos | {N_CLASSES} classes ({len(HEALTHY)} healthy) | train {len(TRAIN):,}  val {len(VAL):,}  test {len(TEST):,}')
print(f'RoCoLe      : {len(ro):,} photos | {len(np.unique(plants))} plants | healthy {int((1 - ro_sick).sum())}, sick {int(ro_sick.sum())} | dev half {DEV.sum()}  locked half {LOCKED.sum()}')

Two **split rules** matter most:

* **PlantVillage is split by leaf.** The same leaf was photographed several times. If sibling photos land in both train and test, accuracy looks too high because the test is partly a memory test. (The split shipped with the dataset has this flaw.)
* **RoCoLe is split by plant** into a **`dev`** half and a **`locked`** half. `dev` is for making choices (a setting, a recipe, a cut-off). `locked` is the final exam: nothing is ever chosen with it.

The next cell proves the splits are clean.

In [ ]:
assert pv_img.shape[0] == len(pv) and ro_img.shape[0] == len(ro) and pv_img.shape[1:] == (224, 224, 3)
leaf_overlap = len(set(pv['gid'].values[TRAIN]) & set(pv['gid'].values[TEST])) + len(set(pv['gid'].values[TRAIN]) & set(pv['gid'].values[VAL]))
plant_overlap = len(set(plants[DEV]) & set(plants[LOCKED]))
assert leaf_overlap == 0 and plant_overlap == 0, 'splits leak!'
print(f'✅ leaves shared between train and val/test: {leaf_overlap} | coffee plants shared between dev and locked: {plant_overlap}')
counts = pv.groupby('cls').size(); print(f'photos per class: min {counts.min()}, median {int(counts.median())}, max {counts.max()} (capped at 200 per class so no class dominates)')

Now **look at the data**. Always do this before training anything.

In [ ]:
def show(imgs, titles, ncols=8, cell=1.9, fs=7):
    nrows = math.ceil(len(imgs) / ncols); fig, ax = plt.subplots(nrows, ncols, figsize=(cell * ncols, cell * nrows * 1.08), squeeze=False)
    for a in ax.ravel(): a.axis('off')
    for a, im, t in zip(ax.ravel(), imgs, titles): a.imshow(im); a.set_title(t, fontsize=fs)
    plt.tight_layout(); plt.show()

rs = np.random.RandomState(1)
lab = rs.choice(TRAIN, 8, replace=False)
heal, sick_ = rs.choice(np.where(~ro_sick.astype(bool))[0], 4, replace=False), rs.choice(np.where(ro_sick.astype(bool))[0], 4, replace=False)
show(list(pv_img[lab]) + list(ro_img[heal]) + list(ro_img[sick_]),
     [pv['cls'].iloc[i].replace('___', '\n').replace('_', ' ')[:30] for i in lab] + ['RoCoLe: ' + ro['cls'].iloc[i] for i in list(heal) + list(sick_)])

**Top: PlantVillage. Below: RoCoLe.** Lab photos have an even backdrop and one centred leaf. Field photos have soil, weeds, shadows and partial leaves.

> ✅ **Checkpoint 2.** 7,629 + 1,560 photos, zero shared leaves or plants, and a clear visual gap. If the assert fails, stop and ask an instructor.

---
## Step 3 · The model: ConvNeXt-Tiny
ConvNeXt-Tiny (about 28 M parameters) is pretrained on ImageNet, so it already knows edges and textures. We swap its last layer for a new 38-way one and **fine-tune** the whole network. The weights come from the data volume, so no internet is needed.

> 💡 **Why this model?** ConvNeXt-Tiny is a modern network that is accurate yet small enough to fine-tune on a 16 GB T4.

In [ ]:
WEIGHTS = DATA / 'weights/convnext_tiny_imagenet1k_v1.pth'

def make_model(num_classes=N_CLASSES):
    m = torchvision.models.convnext_tiny(weights=None)
    m.load_state_dict(torch.load(WEIGHTS, map_location='cpu', weights_only=True))     # ImageNet-pretrained weights
    m.classifier[2] = nn.Linear(768, num_classes)                                       # new head: 768 features -> 38 classes
    return m

m = make_model()
print(f'ConvNeXt-Tiny: {sum(p.numel() for p in m.parameters()) / 1e6:.1f} M parameters | new head: {m.classifier[2]}')
print('stages:', [type(c).__name__ + f'({len(c)})' if hasattr(c, '__len__') else type(c).__name__ for c in m.features])
del m

---
## Step 4 · The training code
Read this once. Later exercises reuse it. Four pieces:

1. **`LeafDataset`** serves (photo, label) pairs, with an optional augmentation.
2. **`DataLoader`** prepares batches on CPU workers while the GPU trains on the previous batch. Photos travel as small `uint8` arrays and are normalised on the GPU.
3. **`fit()`** is the training loop: fp16 forward pass, `GradScaler`, AdamW with warm-up then cosine decay, gradient clipping. It prints validation accuracy each epoch.
4. **`predict_proba()`** turns photos into class probabilities.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1) * 255          # ImageNet statistics (the pretrained model expects them)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1) * 255

def to_gpu(batch_u8):
    """uint8 (B,H,W,3) tensor on the CPU -> normalised float (B,3,H,W) on the device, channels-last (fastest layout for convolutions on tensor cores)."""
    x = batch_u8.to(DEVICE, non_blocking=True).permute(0, 3, 1, 2).float()
    return ((x - MEAN) / STD).contiguous(memory_format=torch.channels_last)

class LeafDataset(torch.utils.data.Dataset):
    """Serves (uint8 photo, label). With `augment`, every photo is served twice per epoch: once untouched, once freshly augmented."""
    def __init__(self, images, labels, idx, augment=None):
        self.images, self.labels, self.augment, self.epoch = images, labels, augment, 0
        self.items = [(int(i), False) for i in idx] + ([(int(i), True) for i in idx] if augment else [])
    def __len__(self): return len(self.items)
    def __getitem__(self, k):
        i, aug = self.items[k]; img = self.images[i]
        if aug: img = self.augment(img, np.random.default_rng([SEED, self.epoch, i]))   # a new random draw each epoch, reproducible from (seed, epoch, photo)
        return torch.from_numpy(np.ascontiguousarray(img)), int(self.labels[i])

@torch.inference_mode()
def predict_proba(model, images, bs=128):
    model.eval(); out = []
    for i in range(0, len(images), bs):
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=CUDA_OK):
            out.append(F.softmax(model(to_gpu(torch.from_numpy(np.ascontiguousarray(images[i:i + bs])))).float(), 1).cpu())
    return torch.cat(out).numpy()

def fit(augment=None, epochs=EPOCHS, lr=LR, batch=BATCH_SIZE, seed=SEED):
    torch.manual_seed(seed)
    model = make_model().to(DEVICE).to(memory_format=torch.channels_last)
    ds = LeafDataset(pv_img, y, TRAIN, augment)
    dl = torch.utils.data.DataLoader(ds, batch_size=batch, shuffle=True, drop_last=True, num_workers=NUM_WORKERS, pin_memory=CUDA_OK,
                                     prefetch_factor=PREFETCH if NUM_WORKERS else None, generator=torch.Generator().manual_seed(seed))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    total, warm = epochs * len(dl), max(10, int(0.1 * epochs * len(dl)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1, s / total))))
    scaler = torch.amp.GradScaler(enabled=CUDA_OK)
    if CUDA_OK: torch.cuda.reset_peak_memory_stats()
    hist, t_start = [], time.time()
    for ep in range(epochs):
        ds.epoch = ep; model.train(); t0 = time.time(); loss_sum = torch.zeros((), device=DEVICE); seen = 0
        for xb, yb in dl:
            x, t = to_gpu(xb), yb.to(DEVICE, non_blocking=True)
            with torch.autocast(DEVICE, dtype=torch.float16, enabled=CUDA_OK):
                loss = F.cross_entropy(model(x), t, label_smoothing=LABEL_SMOOTHING)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)         # clip after unscaling so the threshold is in real units
            scaler.step(opt); scaler.update(); sched.step()
            loss_sum += loss.detach() * len(t); seen += len(t)
        if CUDA_OK: torch.cuda.synchronize()
        secs = time.time() - t0; val_acc = float((predict_proba(model, pv_img[VAL]).argmax(1) == y[VAL]).mean())
        hist.append({'epoch': ep + 1, 'train_loss': float(loss_sum) / seen, 'val_acc': val_acc, 'seconds': secs, 'img_per_s': seen / secs})
        print(f'epoch {ep + 1}/{epochs}  train loss {hist[-1]["train_loss"]:.3f}  val acc {val_acc:.3f}  {secs:5.1f}s  ({seen / secs:,.0f} photos/s)', flush=True)
    info = {'epochs': epochs, 'photos_per_epoch': len(ds), 'train_seconds': time.time() - t_start, 'peak_gpu_gb': torch.cuda.max_memory_allocated() / 2**30 if CUDA_OK else 0.0}
    return model.eval(), pd.DataFrame(hist), info

print('training machinery defined')

**Speed check.** We time a few steps on random data and project the baseline's run time. If it looks too long on your hardware, lower `EPOCHS` and re-run from the knobs cell.

In [ ]:
def gpu_speed_check(steps=12, batch=BATCH_SIZE):
    m = make_model().to(DEVICE).to(memory_format=torch.channels_last); opt = torch.optim.AdamW(m.parameters(), lr=1e-5); scaler = torch.amp.GradScaler(enabled=CUDA_OK)
    x = torch.randn(batch, 3, 224, 224, device=DEVICE).contiguous(memory_format=torch.channels_last); t = torch.randint(0, N_CLASSES, (batch,), device=DEVICE)
    for s in range(steps + 3):
        if s == 3:
            if CUDA_OK: torch.cuda.synchronize()
            t0 = time.time()
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=CUDA_OK): loss = F.cross_entropy(m(x), t)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    if CUDA_OK: torch.cuda.synchronize()
    del m, opt; torch.cuda.empty_cache() if CUDA_OK else None
    return steps * batch / (time.time() - t0)

GPU_IMG_S = gpu_speed_check()
print(f'{DEVICE.upper()} training speed on random data: ~{GPU_IMG_S:,.0f} photos/s')
print(f'projected baseline run : {EPOCHS} epochs x {len(TRAIN):,} photos  = ~{EPOCHS * len(TRAIN) / GPU_IMG_S / 60:.1f} min  (plus a validation pass per epoch)')

---
## Step 5 · The baseline: train on lab photos only
This is the model a team would build first: fine-tune on the lab data, check accuracy on lab photos, ship.

> ✋ **Predict.** Double-click and write your guesses:
>
> * Accuracy on held-out **lab** photos (38 classes, chance is 2.6%): ______ %
> * Ranking score on **coffee** photos (0.5 = coin flip, 1.0 = perfect): ______

In [ ]:
t_run = time.time()
baseline, hist_base, info_base = fit(augment=None)
print(f'\nbaseline trained in {info_base["train_seconds"]:.0f}s | peak GPU memory {info_base["peak_gpu_gb"]:.1f} GiB')

While that trained, `val acc` is accuracy on **validation** photos (held-out leaves from the lab dataset). Now the real **test** with leaves it never saw, still from the lab:

In [ ]:
def triage_score(P): return 1 - P[:, HEALTHY].sum(1)         # P(sick) = 1 - sum of the 12 healthy classes' probabilities

def best_cutoff(s, mask):
    """The cut-off on the sick score that maximises balanced accuracy on the photos selected by `mask`."""
    cuts = np.linspace(.02, .98, 97)
    return float(cuts[np.argmax([balanced_accuracy_score(ro_sick[mask], s[mask] >= c) for c in cuts])])

CUT_KEY = 'cut-off tuned on the dev half'

def evaluate(model):
    P_te, P_ro = predict_proba(model, pv_img[TEST]), predict_proba(model, ro_img)
    s = triage_score(P_ro); halves = (('dev', DEV), ('LOCKED', LOCKED))
    r = {'lab: 38-class accuracy (PlantVillage test)': float((P_te.argmax(1) == y[TEST]).mean())}
    for name, mask in halves: r[f'field {name}: sick-vs-healthy AUC (ranking)'] = roc_auc_score(ro_sick[mask], s[mask])
    for name, mask in halves: r[f'field {name}: balanced accuracy at the default 0.5 cut-off'] = balanced_accuracy_score(ro_sick[mask], s[mask] >= .5)
    for name, mask in halves: r[f'field {name}: share called sick at 0.5 (truth {ro_sick[mask].mean():.0%})'] = float((s[mask] >= .5).mean())
    cut = best_cutoff(s, DEV)                                   # chosen on the dev half only ...
    r[CUT_KEY] = cut
    r['field dev: balanced accuracy at that cut-off'] = balanced_accuracy_score(ro_sick[DEV], s[DEV] >= cut)           # (flattering: tuned on this very half)
    r['field LOCKED: balanced accuracy at that cut-off'] = balanced_accuracy_score(ro_sick[LOCKED], s[LOCKED] >= cut)     # ... and judged on the locked half
    return r, s

def field_table(models):
    """Plain-English table: one row per question, one column per (model, half of the coffee photos)."""
    get = lambda r, half, what: next(v for k, v in r.items() if k.startswith(f'field {half}: {what}'))
    cols = {}
    for name, r in models.items():
        for label, half, mask in (('dev', 'dev', DEV), ('locked', 'LOCKED', LOCKED)):
            cols[(name, label)] = {
                'Ranking score (AUC)  [0.5 = coin flip, 1.0 = perfect]': f"{get(r, half, 'sick-vs-healthy AUC'):.2f}",
                'True share of sick leaves': f'{ro_sick[mask].mean():.0%}',
                'Share the app calls "sick" at cut-off 0.5': f"{get(r, half, 'share called sick'):.0%}",
                'Accuracy at cut-off 0.5  [0.5 = coin flip]': f"{get(r, half, 'balanced accuracy at the default'):.2f}",
                'Tuned cut-off (picked on dev)': f'{r[CUT_KEY]:.2f}' if label == 'dev' else '(same)',
                'Accuracy at the tuned cut-off': f"{get(r, half, 'balanced accuracy at that cut-off'):.2f}"}
    return pd.DataFrame(cols)

res_base, score_base = evaluate(baseline)
print(f"lab test accuracy: {res_base['lab: 38-class accuracy (PlantVillage test)']:.1%}")

## Step 6 · The field test
Same model, now on **real phone photos of coffee leaves**. We only ask: **healthy or sick?**

The table has one row per question, for the two halves of the coffee photos. The notes under it explain each row.

In [ ]:
field_table({'baseline': res_base})

### How to read this table
Each row is a question. The columns are the two halves of the coffee photos: `dev` (we may tune on it) and `locked` (the final exam, so trust this column). The last column is from our reference run (GB10 GPU, seed 42). Yours will be close, not identical.

| Row | What it asks | Reference run |
|---|---|---|
| **Ranking score (AUC)** | Take one sick and one healthy leaf. Does the sick one get the higher sick score? 0.5 = coin flip, 1.0 = always. | 0.85 to 0.86: usually yes |
| **True share of sick leaves** | Reference: how many of the leaves really are sick. | about 50% |
| **Share the app calls "sick" at 0.5** | If the app says "sick" for any score of 0.5 or more, how many leaves does it flag? It should be close to the true share. | 99% to 100%: nearly everything |
| **Accuracy at cut-off 0.5** | How often is that call right? It is averaged over healthy and sick leaves separately, so "always sick" scores 0.5, not a lucky 50%. | 0.50 to 0.51: a coin flip |
| **Tuned cut-off** | A better score line, picked on `dev` to give the best accuracy. | 0.73 |
| **Accuracy at the tuned cut-off** | The same question with the new line. Trust `locked`: `dev` looks better because we tuned on it. | 0.77 on `dev`, 0.79 on `locked` |

**In one line:** the model orders leaves sensibly (ranking 0.85), but its default line sits in the wrong place, so it calls nearly everything sick. Moving the line fixes much of it.

In [ ]:
def score_hist(scores_by_model, cuts=None, mask=LOCKED, title='LOCKED half'):
    fig, ax = plt.subplots(1, len(scores_by_model), figsize=(5.4 * len(scores_by_model), 3.4), squeeze=False)
    for a, (name, s) in zip(ax[0], scores_by_model.items()):
        a.hist(s[mask & (ro_sick == 0)], bins=25, range=(0, 1), alpha=.65, label='healthy leaves', color='#2a9d5c'); a.hist(s[mask & (ro_sick == 1)], bins=25, range=(0, 1), alpha=.65, label='sick leaves', color='#d9625a')
        a.axvline(.5, color='k', lw=1, ls=':', label='default cut-off 0.5')
        if cuts: a.axvline(cuts[name], color='k', lw=1.2, ls='--', label=f'cut-off tuned on dev ({cuts[name]:.2f})')
        a.set_title(f'{name}: sick score, {title}', fontsize=10); a.set_xlabel('sick score (0 = confident healthy, 1 = confident sick)'); a.legend(frameon=False, fontsize=8)
    plt.tight_layout(); plt.show()
score_hist({'baseline': score_base}, cuts={'baseline': res_base[CUT_KEY]})

Each bar counts coffee leaves with that sick score. A useful model puts the red (sick) bars to the right of the green (healthy) ones. A *usable* one also has the cut-off line between them. Where are your bars relative to the two lines?

### Where does the tuned cut-off come from?
We tried many cut-offs, measured accuracy at each one **using only the `dev` half**, and kept the best. The curve below shows that search. The dashed line is the pick. The `locked` curve is only there to judge it: we never chose with it.

In [ ]:
cuts = np.linspace(.02, .98, 97)                                          # every cut-off we try
acc_at = lambda s, mask: [balanced_accuracy_score(ro_sick[mask], s[mask] >= c) for c in cuts]
picked = res_base[CUT_KEY]
MY_CUT = None                                                              # 🔧 try your own line, e.g. 0.65, then re-run this cell

fig, ax = plt.subplots(figsize=(7.5, 3.5))
ax.plot(cuts, acc_at(score_base, DEV), color='#2a6fb0', label='dev (we choose with this)')
ax.plot(cuts, acc_at(score_base, LOCKED), color='#d9625a', label='locked (final exam, never used to choose)')
ax.axvline(.5, color='k', lw=1, ls=':', label='default cut-off 0.5'); ax.axvline(picked, color='k', lw=1.2, ls='--', label=f'picked on dev: {picked:.2f}')
if MY_CUT is not None: ax.axvline(MY_CUT, color='#2a9d5c', lw=1.5, label=f'your line: {MY_CUT}')
ax.set_xlabel('cut-off: call a leaf "sick" if its sick score is at least this'); ax.set_ylabel('accuracy (0.5 = coin flip)'); ax.set_ylim(.4, 1)
ax.legend(frameon=False, fontsize=8, loc='upper left'); plt.tight_layout(); plt.show()
for label, c in (('default 0.5', .5), ('picked on dev', picked)) + ((('your line', MY_CUT),) if MY_CUT is not None else ()):
    print(f'{label:14s} ({c:.2f}):  accuracy on dev {balanced_accuracy_score(ro_sick[DEV], score_base[DEV] >= c):.2f}   on locked {balanced_accuracy_score(ro_sick[LOCKED], score_base[LOCKED] >= c):.2f}')

**How to read this plot**

*The numbers below are from our reference run (GB10 GPU, seed 42). Yours will be close, not identical. Check the shape, not the exact values.*

* At 0.5 accuracy is a coin flip, because every leaf scores above the line.
* It rises as the line moves up, peaks around 0.7, then falls back to 0.5 when the line is so high that every leaf is called healthy.
* The good zone is **narrow**: only lines between about 0.70 and 0.76 reach 0.75 accuracy. The line needs care.
* The `locked` curve peaks in almost the same place as `dev`, so a line chosen on `dev` carries over to photos we did not choose with.

**Look at the mistakes.** Top row: healthy leaves with the highest sick score. Bottom row: sick leaves with the lowest.

In [ ]:
def error_gallery(scores, n=8):
    fa = np.where(LOCKED & (ro_sick == 0))[0]; fa = fa[np.argsort(-scores[fa])][:n]            # healthy, highest sick score
    ms = np.where(LOCKED & (ro_sick == 1))[0]; ms = ms[np.argsort(scores[ms])][:n]               # sick, lowest sick score
    show(list(ro_img[fa]) + list(ro_img[ms]), [f'healthy, score {scores[i]:.2f}' for i in fa] + [f'{ro["cls"].iloc[i]}, score {scores[i]:.2f}' for i in ms], ncols=n)
error_gallery(score_base)

### What the baseline shows
On our reference runs (GB10 GPU; yours will differ a little):

* **Lab photos: excellent.** About 97% correct. By this test alone you would ship it.
* **Field photos: it ranks decently.** Ranking score about 0.85: in most pairs the sick leaf scores higher.
* **But its default decision is useless.** Almost every coffee leaf, healthy or sick, scores above 0.5, so the app says "sick" for nearly everything (accuracy about 0.5, a coin flip). A likely reason: coffee is not one of the 14 trained crops, so an unfamiliar leaf looks "diseased" to the model.
* **A cut-off tuned on `dev` fixes much of it** (accuracy about 0.78 on `locked`). But it fits this one plantation only.

**The lesson:** the lab test could not have told you any of this.

> ✅ **Checkpoint 3.** Lab accuracy in the high 90s, ranking score clearly lower, and nearly every coffee leaf called "sick" at 0.5. Compare with your prediction.

---
## Step 7 · Does it read the leaf or the backdrop?
In PlantVillage each class was mostly photographed in one session, so the **backdrop and lighting can leak the label**. We test your model on three versions of each test photo: the **whole photo**, the **backdrop only** (leaf painted over) and the **leaf only** (backdrop blacked out).

In [ ]:
test_masks = np.unpackbits(np.load(DATA / 'plantvillage/masks.npy')[TEST], axis=-1).astype(bool)

def backdrop_only(img, m):
    grown = np.asarray(Image.fromarray((m * 255).astype('uint8')).filter(ImageFilter.MaxFilter(7))) > 0      # grow the mask a little to cover the leaf edge
    out = img.copy(); out[grown] = img[~m].mean(0) if (~m).any() else img.mean((0, 1)); return out

X_bg = np.stack([backdrop_only(a, m) for a, m in zip(pv_img[TEST], test_masks)])
X_leaf = np.stack([a * m[..., None] for a, m in zip(pv_img[TEST], test_masks)])
show(list(pv_img[TEST][:5]) + list(X_bg[:5]) + list(X_leaf[:5]), ['whole photo'] * 5 + ['backdrop only'] * 5 + ['leaf only'] * 5, ncols=5, cell=2.2)

def probe(model):
    return {'whole photo': float((predict_proba(model, pv_img[TEST]).argmax(1) == y[TEST]).mean()),
            'backdrop only (no leaf!)': float((predict_proba(model, X_bg).argmax(1) == y[TEST]).mean()),
            'leaf only (no backdrop)': float((predict_proba(model, X_leaf).argmax(1) == y[TEST]).mean())}
probe_base = probe(baseline)
print(f'38-class accuracy on lab test photos (chance = {1 / N_CLASSES:.1%}):'); print(pd.Series(probe_base, name='baseline').map('{:.1%}'.format).to_string())

**With no leaf at all, the model still gets 14–27% right** on our reference runs (GB10 GPU; chance is 2.6%). With only the leaf it gets about 80–93%. So the leaf carries most of the signal, but the model also leans on the backdrop. Real fields have no lab backdrop, so it has less to lean on there.

(Treat the numbers as rough: a painted-over leaf also looks strange to the model.)

---
## Step 8 · A first manual fix: augmentation
If lab photos are too clean, show the model each training photo again with random changes that mimic the field. We chose a recipe beforehand, using only the `dev` half of the coffee photos. It is called `flip+rrc0.1+photo1` and has three ingredients applied in order:

| Ingredient | What it does | Why |
|---|---|---|
| `flip` | random 90° rotation and mirror | phone photos have no fixed "up" |
| `rrc0.1` | random zoom into at least 10% of the photo | field photos are close-ups and partial leaves |
| `photo1` | random brightness, contrast, colour, exposure, blur | sun, shade and a cheap lens |

**Does it help the fine-tuned ConvNeXt? That is what you measure now.**

The whole recipe is about 25 lines of code:

In [ ]:
def _flip_rot90(a, rng):
    a = np.rot90(a, int(rng.integers(0, 4)))
    return a[:, ::-1] if rng.random() < .5 else a

def random_resized_crop(img, rng, min_area=.25, ratio=(3 / 4, 4 / 3)):
    H = W = img.shape[0]; area = H * W * rng.uniform(min_area, 1.0); ar = np.exp(rng.uniform(np.log(ratio[0]), np.log(ratio[1])))
    w, h = min(W, int(round(np.sqrt(area * ar)))), min(H, int(round(np.sqrt(area / ar))))
    x0, y0 = int(rng.integers(0, W - w + 1)), int(rng.integers(0, H - h + 1))
    return np.asarray(Image.fromarray(np.ascontiguousarray(img)).crop((x0, y0, x0 + w, y0 + h)).resize((W, H), Image.BICUBIC))

def photometric(img, rng, s=1.0):
    im = Image.fromarray(np.ascontiguousarray(img))
    im = ImageEnhance.Brightness(im).enhance(1 + rng.uniform(-.45, .45) * s); im = ImageEnhance.Contrast(im).enhance(1 + rng.uniform(-.4, .4) * s)
    im = ImageEnhance.Color(im).enhance(1 + rng.uniform(-.4, .4) * s)
    h, sat, v = im.convert('HSV').split(); h = Image.fromarray(((np.asarray(h).astype(int) + int(rng.uniform(-10, 10) * s)) % 256).astype('uint8'))
    im = Image.merge('HSV', (h, sat, v)).convert('RGB')
    a = np.asarray(im).astype('float32') / 255; a = a ** rng.uniform(1 - .35 * s, 1 + .35 * s)                  # gamma (exposure)
    a *= np.array([1 + rng.uniform(-.12, .12) * s, 1, 1 + rng.uniform(-.12, .12) * s])                           # colour temperature (warmer / cooler)
    im = Image.fromarray((np.clip(a, 0, 1) * 255).astype('uint8'))
    if rng.random() < .5 * s: im = im.filter(ImageFilter.GaussianBlur(rng.uniform(.3, 1.3)))                     # soft focus
    return np.asarray(im)

def apply_recipe(img, rng, recipe):
    """recipe = ingredients joined by '+', applied in order: flip | rrcS (min crop area S) | photoS (jitter strength S)."""
    for op in recipe.split('+'):
        if op == 'flip': img = _flip_rot90(img, rng)
        elif op.startswith('rrc'): img = random_resized_crop(img, rng, min_area=float(op[3:]))
        elif op.startswith('photo'): img = photometric(img, rng, float(op[5:]))
        elif op != 'none': raise ValueError(f'unknown ingredient {op!r} (this notebook knows flip, rrcS, photoS)')
    return img

See it work: each row is one training photo; each column adds one more ingredient.

In [ ]:
steps_ = ['none', 'flip', 'flip+rrc0.1', 'flip+rrc0.1+photo1', 'flip+rrc0.1+photo1', 'flip+rrc0.1+photo1']
demo = np.random.RandomState(3).choice(TRAIN, 3, replace=False)
show([apply_recipe(pv_img[i], np.random.default_rng(100 * c + int(i)), r) for i in demo for c, r in enumerate(steps_)],
     [(r if c < 4 else f'full recipe, draw {c - 2}') for _ in demo for c, r in enumerate(steps_)], ncols=len(steps_), cell=2.2)

> 🔧 **Try it.** Change `RECIPE` in the knobs cell (`'flip+rrc0.5+photo1'` is a milder zoom, `'flip+photo1'` has none) and re-run from Step 8.

The augmentation runs on the **CPU** while the GPU trains. If the CPUs are too slow, the GPU waits. Quick check:

In [ ]:
t0 = time.time(); [apply_recipe(pv_img[i], np.random.default_rng(i), RECIPE) for i in TRAIN[:200]]; per_worker = 200 / (time.time() - t0)
print(f'one CPU worker makes ~{per_worker:,.0f} augmented photos/s -> {NUM_WORKERS} workers ~{per_worker * NUM_WORKERS:,.0f}/s (the GPU consumes ~{GPU_IMG_S:,.0f}/s)')
print('data pipeline can keep the GPU busy ✅' if per_worker * NUM_WORKERS >= GPU_IMG_S * 0.5 else 'CPU-bound ⚠️ the GPU will wait for data: more workers or a bigger CPU request would speed this up (an infrastructure lesson!)')

Now re-train with the recipe: **same model, settings and seed**. Only the data changes. Each photo appears twice per epoch, once clean and once freshly augmented, so this run takes about twice as long.

In [ ]:
augmented, hist_aug, info_aug = fit(augment=lambda img, rng: apply_recipe(img, rng, RECIPE))
print(f'\naugmented model trained in {info_aug["train_seconds"]:.0f}s | peak GPU memory {info_aug["peak_gpu_gb"]:.1f} GiB')

### Compare: same field test, two models
Same table as before, one pair of columns per model. Compare the `locked` columns.

In [ ]:
res_aug, score_aug = evaluate(augmented)
probe_aug = probe(augmented)
BD = 'backdrop only (no leaf!)'
print(f"Lab photos, 38-class accuracy:  baseline {res_base['lab: 38-class accuracy (PlantVillage test)']:.1%}   augmented {res_aug['lab: 38-class accuracy (PlantVillage test)']:.1%}")
print(f"Leaf hidden (backdrop only):    baseline {probe_base[BD]:.1%}   augmented {probe_aug[BD]:.1%}   (lower = relies less on the backdrop)")
field_table({'baseline': res_base, 'augmented': res_aug})

In [ ]:
score_hist({'baseline': score_base, 'augmented': score_aug}, cuts={'baseline': res_base[CUT_KEY], 'augmented': res_aug[CUT_KEY]})

Is the change real, or luck of which plants are in the locked half? We **bootstrap over plants**: re-draw plants at random many times and recompute the gain. The spread is a 95% interval. If it excludes zero, the gain is unlikely to be chance **for these plants**.

In [ ]:
def bootstrap_gain(s_new, s_old, mask, n=1000, seed=0):
    rng = np.random.RandomState(seed); lp = np.unique(plants[mask]); by = {p: np.where((plants == p) & mask)[0] for p in lp}; d = []
    for _ in range(n):
        idx = np.concatenate([by[p] for p in rng.choice(lp, len(lp))]); d.append(roc_auc_score(ro_sick[idx], s_new[idx]) - roc_auc_score(ro_sick[idx], s_old[idx]))
    return float(np.mean(d)), np.percentile(d, [2.5, 97.5])
for name, mask in (('dev', DEV), ('LOCKED', LOCKED)):
    g, (lo, hi) = bootstrap_gain(score_aug, score_base, mask)
    print(f'field AUC gain from augmentation, {name:6s} half: {g:+.3f}   95% CI [{lo:+.3f}, {hi:+.3f}]  ' + ('-> clearly above zero' if lo > 0 else '-> cannot rule out zero' if hi >= 0 else '-> clearly negative'))
error_gallery(score_aug)

### What to take from this
* **The interval covers plant luck, not training luck.** A different random seed can change the answer more than this interval shows.
* **Judge several numbers together:** lab accuracy, ranking score, share called sick, accuracy at the tuned cut-off.
* **One run is a hint, not a verdict.** Step 9 shows how much runs vary.
* The recipe was chosen on `dev`, so `locked` is the fairer judge.
* One field, one phone, one annotator: this is not proof the model is ready for farmers.

> ✅ **Checkpoint 4.** You have two trained models, a comparison table and a bootstrap interval, and you know what that interval does *not* cover.

---
## Step 9 · Scorecard, cost, and saving your work

In [ ]:
cost = pd.DataFrame({n: {'epochs': i['epochs'], 'photos per epoch': i['photos_per_epoch'], 'training time (s)': round(i['train_seconds']),
                         'photos/s (mean)': round(h['img_per_s'].mean()), 'peak GPU memory (GiB)': round(i['peak_gpu_gb'], 1)}
                     for n, (h, i) in {'baseline': (hist_base, info_base), 'augmented': (hist_aug, info_aug)}.items()})
print('What it cost:'); print(cost.to_string())
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for n, h in (('baseline', hist_base), ('augmented', hist_aug)): ax[0].plot(h.epoch, h.train_loss, marker='o', label=n); ax[1].plot(h.epoch, h.val_acc, marker='o', label=n)
ax[0].set_title('training loss'); ax[1].set_title('validation accuracy (lab photos)')
for a in ax: a.set_xlabel('epoch'); a.set_xticks(hist_base.epoch); a.legend(frameon=False)
plt.tight_layout(); plt.show()

Save both models and the numbers to your home folder. It should live on your workspace volume, so it survives a restart (ask if unsure). Later exercises run training as **batch jobs** that cannot see this notebook's memory, so the model must be on disk with everything needed to reload it.

In [ ]:
OUT = Path(os.environ.get('CROPGUARD_OUT', Path.home() / 'cropguard-artifacts')) / 'exercise1'; OUT.mkdir(parents=True, exist_ok=True)
for name, model, res in (('baseline', baseline, res_base), ('augmented', augmented, res_aug)):
    torch.save({'state_dict': model.state_dict(), 'arch': 'convnext_tiny', 'classes': CLASSES, 'healthy_class_ids': HEALTHY.tolist(),
                'recipe': None if name == 'baseline' else RECIPE, 'metrics': res}, OUT / f'{name}.pt')
(OUT / 'results.json').write_text(json.dumps({'knobs': dict(SEED=SEED, EPOCHS=EPOCHS, BATCH_SIZE=BATCH_SIZE, LR=LR, WEIGHT_DECAY=WEIGHT_DECAY, LABEL_SMOOTHING=LABEL_SMOOTHING, RECIPE=RECIPE),
                                              'baseline': res_base, 'augmented': res_aug, 'cost': cost.to_dict()}, indent=1, default=float))
print({p.name: f'{p.stat().st_size / 1e6:.0f} MB' for p in OUT.iterdir()}, '->', OUT)

# A running log of every full run you make, so you can compare seeds (QUICK smoke-test runs are not logged).
AUC = 'field LOCKED: sick-vs-healthy AUC (ranking)'
if not QUICK:
    with open(OUT / 'runs.jsonl', 'a') as f: f.write(json.dumps({'seed': SEED, 'recipe': RECIPE, 'epochs': EPOCHS, 'baseline': res_base, 'augmented': res_aug}, default=float) + '\n')

### How noisy is one run?
Training is random: shuffling, the new layer's starting weights and every augmentation draw depend on the seed. We repeated this notebook with six seeds on the GB10 GPU. Ranking score on the **locked** half:

| seed | baseline | + augmentation | difference |
|---|---|---|---|
| 42 (the default) | 0.862 | 0.870 | +0.008 |
| 7 | 0.917 | 0.918 | +0.001 |
| 123 | 0.811 | 0.890 | +0.079 |
| 1 | 0.754 | 0.803 | +0.049 |
| 2 | 0.849 | 0.898 | +0.049 |
| 3 | 0.895 | 0.901 | +0.006 |
| **mean ± sd** | **0.848 ± 0.059** | **0.880 ± 0.041** | **+0.032 ± 0.032** |

* **The baseline alone ranges from 0.75 to 0.92.** That spread is bigger than the effect we measure, so one before/after pair cannot settle it.
* **On average the recipe helps a little:** about +0.03 on `locked` (better in all 6 seeds) and +0.04 on `dev` (4 of 6).
* **Lab accuracy did not change** (97.3% vs 97.2%). At 0.5, both still call about 97% of leaves "sick". A cut-off tuned on `dev` gives accuracy of about 0.78 (baseline) and 0.81 (augmented).

> 🔧 **Try it.** Set `SEED` to 7 (or any number) and re-run from the knobs cell. The cell below logs every full run. In class, each pair can use a different seed and we pool the "difference" column.

In [ ]:
runs = [json.loads(l) for l in open(OUT / 'runs.jsonl')] if (OUT / 'runs.jsonl').exists() else []
if runs:
    ledger = pd.DataFrame({'seed': [r['seed'] for r in runs], 'baseline AUC': [r['baseline'][AUC] for r in runs], 'augmented AUC': [r['augmented'][AUC] for r in runs]}).drop_duplicates('seed', keep='last').set_index('seed')
    ledger['difference'] = ledger['augmented AUC'] - ledger['baseline AUC']
    print(f'{len(ledger)} full run(s) logged in {OUT / "runs.jsonl"}'); display(ledger.round(3))
    if len(ledger) >= 3: print(f'mean difference {ledger.difference.mean():+.3f}, sd {ledger.difference.std():.3f}')
else: print('no full runs logged yet (QUICK runs are not logged)')

---
## Reflection (5 minutes with your neighbour)
1. **Your workspace.** You had a GPU, a data volume and a ready Python environment in minutes, in a pod you can throw away. What would setting that up by hand cost? What survives if the pod is deleted?
2. **The test decides the story.** Lab accuracy was excellent but field behaviour was poor. What data would you need to trust this model? Who would collect it?
3. **Manual tuning.** You changed one thing and waited minutes for one noisy answer. The knobs cell has seven more (epochs, learning rate, weight decay, label smoothing, batch size, crop and jitter strength). How long would a careful search take by hand?
4. **Noise.** How sure were you about the augmentation effect after one run? How many runs would you want, and how would you run them?
5. **Ranking is not deciding.** Who should choose the cut-off: the data scientist, the agronomist, the app team? What does a wrong answer cost a farmer each way (a missed disease vs needless spraying)?

## What's next
| Exercise | Kubeflow piece | Question |
|---|---|---|
| **2** | **Katib** | Which settings are best? Try many in parallel, 1 GPU each. |
| **3** | **Trainer** | Train the winner faster on 2 GPUs and save the model to a volume. |
| **4** | **Pipelines** | Make it repeatable: prepare → train → evaluate → gate. |
| **5** | **KServe** | Serve the saved model over HTTP. |

The functions from Steps 4 and 8 (`make_model`, `LeafDataset`, `fit`, `apply_recipe`, `evaluate`) are the code the later exercises reuse.

## If something goes wrong
| Symptom | Fix |
|---|---|
| `CUDA_OK` is `False` | No GPU on this notebook server. Ask an instructor. (The notebook falls back to a tiny CPU smoke test only.) |
| `CUDA out of memory` | Set `BATCH_SIZE` to 32 or 16 and re-run from the knobs cell. Or *Kernel → Shut Down All Kernels* in your other notebooks. |
| `Could not find the CropGuard data volume` | The PVC is mounted somewhere unusual. Run `!df -h`, find the folder with `MANIFEST.json`, set `CROPGUARD_DATA`. |
| `DataLoader worker ... exited` / `bus error` | Too little shared memory or RAM. Set `NUM_WORKERS = 2` (or `0`) and re-run from the knobs cell. |
| Training is slower than projected | CPUs are the bottleneck (Step 8 check) or another job shares the GPU. Check `nvidia-smi` in a terminal. |
| Kernel dies loading the data | Memory limit too low for about 1.5 GB of arrays. Tell an instructor. |
| Want to start over | *Kernel → Restart Kernel and Run All Cells*. The notebook only reads the data volume, so it cannot damage it. |